In [1]:
# CELL 1 — Imports and paths

from pathlib import Path
import pandas as pd
import numpy as np
import re

DATA_DIR = Path(
    r"C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\Infinum_2026_Candidate_Dataset_Pack"
)

OUTPUT_DIR = DATA_DIR.parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Notebook 03 started")
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

Notebook 03 started
Data directory: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\Infinum_2026_Candidate_Dataset_Pack
Output directory: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs


In [2]:
# CELL 2 — Load datasets

patients = pd.read_csv(
    DATA_DIR / "patient_360_reference.csv"
)

tele = pd.read_csv(
    DATA_DIR / "teleconsultations.csv"
)

outcomes = pd.read_csv(
    DATA_DIR / "episode_outcomes.csv"
)

linkage = pd.read_csv(
    OUTPUT_DIR / "teleconsultation_linkage.csv"
)

print("Patients:", patients.shape)
print("Teleconsultations:", tele.shape)
print("Outcomes:", outcomes.shape)
print("Linkage:", linkage.shape)

Patients: (5000, 14)
Teleconsultations: (5516, 23)
Outcomes: (5516, 6)
Linkage: (1635, 6)


In [3]:
# CELL 3 — Normalization

def normalize_text(value):
    if pd.isna(value):
        return ""
    
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]", "", value)
    
    return value


print("Normalization function ready")

Normalization function ready


In [4]:
# CELL 4 — Recreate exact teleconsultation linkage

tele_temp = tele.copy()
patients_temp = patients.copy()

tele_temp["name_norm"] = (
    tele_temp["patient_name"]
    .apply(normalize_text)
)

tele_temp["village_norm"] = (
    tele_temp["village"]
    .apply(normalize_text)
)

patients_temp["name_norm"] = (
    patients_temp["canonical_name"]
    .apply(normalize_text)
)

patients_temp["village_norm"] = (
    patients_temp["village"]
    .apply(normalize_text)
)

exact_map = tele_temp.merge(
    patients_temp[
        [
            "patient_id",
            "name_norm",
            "village_norm"
        ]
    ],
    on=[
        "name_norm",
        "village_norm"
    ],
    how="inner"
)

exact_map = exact_map[
    [
        "teleconsult_id",
        "patient_id"
    ]
].drop_duplicates(
    subset="teleconsult_id"
)

print("Exact mappings:", len(exact_map))

Exact mappings: 4620


In [5]:
# CELL 5 — Load fuzzy linkage

fuzzy_map = linkage[
    [
        "teleconsult_id",
        "predicted_patient_id"
    ]
].copy()

fuzzy_map.rename(
    columns={
        "predicted_patient_id": "patient_id"
    },
    inplace=True
)

fuzzy_map = fuzzy_map[
    [
        "teleconsult_id",
        "patient_id"
    ]
].drop_duplicates(
    subset="teleconsult_id"
)

print("Fuzzy mappings:", len(fuzzy_map))

Fuzzy mappings: 1635


In [6]:
# CELL 6 — Complete teleconsultation → patient mapping

tele_patient_map = pd.concat(
    [
        exact_map,
        fuzzy_map
    ],
    ignore_index=True
)

tele_patient_map = (
    tele_patient_map
    .drop_duplicates(
        subset="teleconsult_id",
        keep="first"
    )
)

print(
    "Unique mapped teleconsultations:",
    len(tele_patient_map)
)

print(
    "Total teleconsultations:",
    len(tele)
)

print(
    "Unmapped:",
    len(tele) - len(tele_patient_map)
)

Unique mapped teleconsultations: 5516
Total teleconsultations: 5516
Unmapped: 0


In [7]:
# CELL 7 — Create episode base

episode_base = (
    tele
    .merge(
        tele_patient_map,
        on="teleconsult_id",
        how="left"
    )
    .merge(
        outcomes,
        on="teleconsult_id",
        how="left",
        suffixes=("", "_outcome")
    )
)

episode_base["consult_date"] = pd.to_datetime(
    episode_base["consult_date"],
    errors="coerce"
)

print("Episode base:", episode_base.shape)

print(
    "Mapped:",
    episode_base["patient_id"].notna().sum()
)

print(
    "Unmapped:",
    episode_base["patient_id"].isna().sum()
)

Episode base: (5516, 29)
Mapped: 5516
Unmapped: 0


In [8]:
# CELL 8 — Add Patient 360 context

patient_context = patients[
    [
        "patient_id",
        "age_as_of_2026",
        "gender",
        "preferred_language",
        "known_ncd_status",
        "vulnerability_group",
        "village",
        "block",
        "district",
        "household_id"
    ]
].copy()

features = episode_base.merge(
    patient_context,
    on="patient_id",
    how="left",
    suffixes=("", "_patient")
)

print("Feature base:", features.shape)

Feature base: (5516, 38)


In [9]:
# CELL 9 — Basic patient and consultation features

features["age"] = pd.to_numeric(
    features["age_as_of_2026"],
    errors="coerce"
)

features["distance_km"] = pd.to_numeric(
    features["distance_to_facility_km"],
    errors="coerce"
)

features["duration_min"] = pd.to_numeric(
    features["duration_min"],
    errors="coerce"
)

features["review_due_days"] = pd.to_numeric(
    features["review_due_days"],
    errors="coerce"
)

features["medicine_required"] = (
    features["medicine_advised"]
    .astype(str)
    .str.lower()
    .eq("yes")
    .astype(int)
)

features["test_required"] = (
    features["test_advised"]
    .astype(str)
    .str.lower()
    .eq("yes")
    .astype(int)
)

features["review_required"] = (
    features["review_advised"]
    .astype(str)
    .str.lower()
    .eq("yes")
    .astype(int)
)

print("Basic features created")

Basic features created


In [10]:
# CELL 10 — Care complexity

features["care_requirements_count"] = (
    features["medicine_required"]
    + features["test_required"]
    + features["review_required"]
)

features["complex_care_episode"] = (
    features["care_requirements_count"] >= 2
).astype(int)

print(
    features[
        [
            "care_requirements_count",
            "complex_care_episode"
        ]
    ].value_counts()
)

care_requirements_count  complex_care_episode
2                        1                       2610
1                        0                       1633
3                        1                       1038
0                        0                        235
Name: count, dtype: int64


In [11]:
# CELL 11 — Temporal features

features["consult_year"] = (
    features["consult_date"].dt.year
)

features["consult_month"] = (
    features["consult_date"].dt.month
)

features["consult_dayofweek"] = (
    features["consult_date"].dt.dayofweek
)

features["consult_day"] = (
    features["consult_date"].dt.day
)

features["is_weekend"] = (
    features["consult_dayofweek"] >= 5
).astype(int)

print("Temporal features created")

Temporal features created


In [12]:
# CELL 12 — Historical teleconsultation behaviour

history = (
    features[
        [
            "teleconsult_id",
            "patient_id",
            "consult_date"
        ]
    ]
    .sort_values(
        [
            "patient_id",
            "consult_date"
        ]
    )
    .copy()
)

history["previous_teleconsultations"] = (
    history
    .groupby("patient_id")
    .cumcount()
)

history["has_previous_teleconsult"] = (
    history["previous_teleconsultations"] > 0
).astype(int)

features = features.merge(
    history[
        [
            "teleconsult_id",
            "previous_teleconsultations",
            "has_previous_teleconsult"
        ]
    ],
    on="teleconsult_id",
    how="left"
)

print("Historical consultation features created")

Historical consultation features created


In [13]:
# CELL 13 — Historical consultation frequency

first_consult = (
    features
    .groupby("patient_id")["consult_date"]
    .transform("min")
)

features["days_since_first_recorded_consult"] = (
    features["consult_date"] - first_consult
).dt.days

features["consultation_frequency"] = (
    features["previous_teleconsultations"] /
    features["days_since_first_recorded_consult"].clip(lower=1)
)

features["consultation_frequency"] = (
    features["consultation_frequency"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Frequency features created")

Frequency features created


In [14]:
# CELL 14 — Historical care complexity

features["previous_care_requirements"] = (
    features
    .groupby("patient_id")[
        "care_requirements_count"
    ]
    .transform(
        lambda x: x.shift(1).fillna(0).cumsum()
    )
)

print("Historical care complexity created")

Historical care complexity created


In [15]:
# CELL 15 — Distance bands

features["distance_band"] = pd.cut(
    features["distance_km"],
    bins=[
        -np.inf,
        2,
        5,
        10,
        np.inf
    ],
    labels=[
        "0-2km",
        "2-5km",
        "5-10km",
        "10km+"
    ]
)

print(
    features["distance_band"]
    .value_counts(dropna=False)
)

distance_band
10km+     4377
5-10km     797
2-5km      286
0-2km       56
Name: count, dtype: int64


In [16]:
# CELL 16 — Operational indicators

features["poor_connectivity"] = (
    features["connectivity_quality"]
    .astype(str)
    .str.lower()
    .eq("poor")
    .astype(int)
)

features["long_distance"] = (
    features["distance_km"] > 10
).astype(int)

features["low_digital_access"] = (
    features["vulnerability_group"]
    .astype(str)
    .str.lower()
    .eq("low digital access")
    .astype(int)

)

features["elderly_living_alone"] = (
    features["vulnerability_group"]
    .astype(str)
    .str.lower()
    .eq("elderly living alone")
    .astype(int)
)

print("Operational indicators created")

Operational indicators created


In [17]:
# CELL 17 — Temporal safety check

print("Prediction point:", "End of teleconsultation")

print("\nFeatures based on:")
print("- Current consultation information")
print("- Patient reference information")
print("- Historical consultations before current consultation")
print("- Rural operational context")
print("- Calendar information")

print("\nFuture outcome fields excluded from model features:")
print("- lost_to_followup_label")
print("- dropout_stage_label")

Prediction point: End of teleconsultation

Features based on:
- Current consultation information
- Patient reference information
- Historical consultations before current consultation
- Rural operational context
- Calendar information

Future outcome fields excluded from model features:
- lost_to_followup_label
- dropout_stage_label


In [18]:
# CELL 18 — Final model feature selection

feature_columns = [
    "age",
    "distance_km",
    "duration_min",
    "review_due_days",

    "medicine_required",
    "test_required",
    "review_required",
    "care_requirements_count",
    "complex_care_episode",

    "previous_teleconsultations",
    "has_previous_teleconsult",
    "days_since_first_recorded_consult",
    "consultation_frequency",
    "previous_care_requirements",

    "poor_connectivity",
    "long_distance",
    "low_digital_access",
    "elderly_living_alone",

    "consult_month",
    "consult_dayofweek",
    "is_weekend"
]

print("Number of model features:", len(feature_columns))

for column in feature_columns:
    print(column)

Number of model features: 21
age
distance_km
duration_min
review_due_days
medicine_required
test_required
review_required
care_requirements_count
complex_care_episode
previous_teleconsultations
has_previous_teleconsult
days_since_first_recorded_consult
consultation_frequency
previous_care_requirements
poor_connectivity
long_distance
low_digital_access
elderly_living_alone
consult_month
consult_dayofweek
is_weekend


In [19]:
# CELL 19 — Prepare clean model dataset

model_features = features[
    [
        "episode_id",
        "teleconsult_id",
        "patient_id",
        "consult_date",
        "cohort",
        "lost_to_followup_label"
    ]
    + feature_columns
].copy()

print("Model feature dataset:", model_features.shape)

display(model_features.head())

Model feature dataset: (5516, 27)


,episode_id,teleconsult_id,patient_id,consult_date,cohort,lost_to_followup_label,age,distance_km,duration_min,review_due_days,...,days_since_first_recorded_consult,consultation_frequency,previous_care_requirements,poor_connectivity,long_distance,low_digital_access,elderly_living_alone,consult_month,consult_dayofweek,is_weekend
0,E0000001,TC0000001,P000003,2026-05-19,DEVELOPMENT,0.0,47,29.7,15,NaN,...,30,0.033333,0.0,1,1,1,0,5,1,0
1,E0000002,TC0000002,P000004,2026-07-22,EVALUATION,NaN,45,33.3,18,21.0,...,87,0.022989,0.0,0,1,0,0,7,2,0
2,E0000003,TC0000003,P000004,2026-08-11,EVALUATION,NaN,45,27.6,20,7.0,...,107,0.028037,1.0,0,1,0,0,8,1,0
3,E0000004,TC0000004,P000005,2026-06-02,DEVELOPMENT,0.0,46,25.7,16,30.0,...,0,0.000000,0.0,0,1,0,1,6,1,0
4,E0000005,TC0000005,P000006,2026-04-24,DEVELOPMENT,1.0,18,17.2,19,14.0,...,0,0.000000,0.0,0,1,0,0,4,4,0


In [20]:
# CELL 20 — Missing-value check

missing = (
    model_features[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values in model features")
else:
    print("Missing values:")
    display(missing)

Missing values:


review_due_days    1763
dtype: int64

In [21]:
# CELL 21 — Handle numeric missing values

for column in feature_columns:
    if pd.api.types.is_numeric_dtype(
        model_features[column]
    ):
        model_features[column] = (
            model_features[column]
            .fillna(
                model_features[column].median()
            )
        )

print("Numeric missing values handled")

Numeric missing values handled


In [22]:
# CELL 22 — Final feature validation

print("Rows:", len(model_features))
print("Features:", len(feature_columns))

print(
    "Remaining missing values:",
    model_features[feature_columns].isna().sum().sum()
)

print(
    "Development rows:",
    (
        model_features["cohort"] == "DEVELOPMENT"
    ).sum()
)

print(
    "Evaluation rows:",
    (
        model_features["cohort"] == "EVALUATION"
    ).sum()
)

print(
    "Target available:",
    model_features["lost_to_followup_label"]
    .notna()
    .sum()
)

Rows: 5516
Features: 21
Remaining missing values: 0
Development rows: 4132
Evaluation rows: 1384
Target available: 4132


In [23]:
# CELL 23 — Save final feature dataset

feature_path = (
    OUTPUT_DIR /
    "model_features.csv"
)

model_features.to_csv(
    feature_path,
    index=False
)

print("Saved:", feature_path)

Saved: C:\Users\RAAJARAPU SUSWIN\Downloads\case-a-thon_dset\outputs\model_features.csv


In [24]:
# CELL 24 — Save feature metadata

feature_metadata = pd.DataFrame({
    "feature": feature_columns,
    "description": [
        "Patient age at reference date",
        "Distance to facility in km",
        "Teleconsultation duration",
        "Review due interval in days",
        "Medicine required during current episode",
        "Test required during current episode",
        "Review required during current episode",
        "Number of care requirements",
        "Episode requiring at least two care actions",
        "Number of previous teleconsultations",
        "Whether patient had a previous teleconsultation",
        "Days since first recorded consultation",
        "Historical consultation frequency",
        "Cumulative previous care requirements",
        "Poor recorded connectivity",
        "Distance greater than 10 km",
        "Low digital access vulnerability",
        "Elderly living alone vulnerability",
        "Consultation month",
        "Day of week",
        "Weekend indicator"
    ]
})

feature_metadata.to_csv(
    OUTPUT_DIR / "feature_metadata.csv",
    index=False
)

print("Feature metadata saved")

Feature metadata saved


In [25]:
# CELL 25 — Notebook 03 final summary

print("=" * 55)
print("NOTEBOOK 03 COMPLETE")
print("=" * 55)

print(
    "Total episodes:",
    len(model_features)
)

print(
    "Development:",
    (
        model_features["cohort"]
        == "DEVELOPMENT"
    ).sum()
)

print(
    "Evaluation:",
    (
        model_features["cohort"]
        == "EVALUATION"
    ).sum()
)

print(
    "Model features:",
    len(feature_columns)
)

print(
    "Missing feature values:",
    model_features[
        feature_columns
    ].isna().sum().sum()
)

print("\nSaved:")
print("model_features.csv")
print("feature_metadata.csv")

NOTEBOOK 03 COMPLETE
Total episodes: 5516
Development: 4132
Evaluation: 1384
Model features: 21
Missing feature values: 0

Saved:
model_features.csv
feature_metadata.csv
